In [1]:
from pyTNG import utils
from pyTNG import data_interface as _data_interface
import os
import pandas as pd
import numpy as np
import illustris_python as il
from pyTNG.spectra import StarSpectrumFactory
from pyTNG import spectra
import matplotlib.pyplot as plt
import time
import astropy.units as u
import scipy.integrate as integ
import pyTNG.utils as utils
from pyTNG.cosmology import TNGcosmo

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
h = TNGcosmo.h
snap_num = 13

In [4]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:80% !important; }</style>"))

In [5]:
basepath = "/virgo/simulations/IllustrisTNG/"
sim_name = 'L35n2160TNG'
sim = _data_interface.TNG50Simulation(os.path.join(basepath, sim_name))

In [ ]:
dataset = next(sim.group_cat[snap_num].chunk_generator('subhalo'))

In [ ]:
keys_needed = ['SubhaloGasMetallicity', 'SubhaloGasMetallicityHalfRad', 'SubhaloHalfmassRadType', 'SubhaloMassInHalfRad', 
               'SubhaloMassInHalfRadType', 'SubhaloMassInRad', 'SubhaloMassInRadType', 'SubhaloSFRinHalfRad', 
               'SubhaloSFRinRad', 'SubhaloPos']

In [ ]:
sub_dict = {key: dataset[key] for key in keys_needed}

In [ ]:
dataset_df = utils.dfFromArrDict(sub_dict)

In [ ]:
columns_of_interest = [('SubhaloGasMetallicity', 0), 
                       ('SubhaloGasMetallicityHalfRad',0),
                       ('SubhaloHalfmassRadType', 4),
                       ('SubhaloMassInHalfRad', 0),
                       ('SubhaloMassInHalfRadType', 4),
                       ('SubhaloMassInRad', 0),
                       ('SubhaloMassInRadType', 0),
                       ('SubhaloMassInRadType', 4),
                       ('SubhaloSFRinHalfRad', 0),
                       ('SubhaloSFRinRad', 0),
                       ('SubhaloPos', 0),
                       ('SubhaloPos', 1),
                       ('SubhaloPos', 2)
                      ]

Remove all halos that do not have any star particles or star formation rate

In [ ]:
def reduce_df(df):
    filt = (df[('SubhaloMassInRadType', 4)]>0) & (df[('SubhaloSFRinRad', 0)]>0)
    new_df = df[filt]
    return new_df

In [ ]:
df_to_save = dataset_df[columns_of_interest]
df_to_save = reduce_df(df_to_save)

In [ ]:
df_to_save.to_pickle(path)

In [7]:
df = pd.read_pickle(path)

In [8]:
z_start = sim.snap_cat[snap_num].header['Redshift']

In [9]:
sim_path = os.path.join(basepath, sim_name, 'output')

In [10]:
bpp = [
    '/ptmp/mpa/mglatzle/TNG_f_esc/BPASSv2.2.1_release-07-18-Tuatara/',
    '2.2.1',
    'chab300',
    'bin']
lam_min = None
lam_max = None
specFac = StarSpectrumFactory(bpp)

In [11]:
def get_stellar_dist(stars, df, index):
    gal_center = np.array([df.loc[index][('SubhaloPos',0)],df.loc[index][('SubhaloPos',1)],df.loc[index][('SubhaloPos',2)]])
    rel_pos = stars['Coordinates']-gal_center
    radius = df.loc[index][('SubhaloHalfmassRadType',4)]*2
    dist = np.sqrt(np.sum(np.square(rel_pos), axis=1))
    rel_dist = dist/radius
    stars['rel_dist'] = rel_dist
    return

In [30]:
def add_lum_ion(df, sim_path, snap_num):
    luminosities = []
    ion_lums = []
    df[('ion_lum',0)] = np.nan
    for idx in df.index:
        stars = il.snapshot.loadSubhalo(sim_path, snap_num, idx,'stars')
        utils.dropWindParticles(stars)
        stars['Redshift'] = z_start
        stars['ages'] = spectra._computeAgeFromFormationTime(stars['Redshift'],stars['GFM_StellarFormationTime'])
        get_stellar_dist(stars, df, idx)
        idces = stars['rel_dist']<1
        utils._keepPartsByIdx(stars, idces)
        specFac.computeStellarSpectra(stars, Q_0=True)
        ion_lum = stars['Q_0'].sum()
        summed_spectrum = stars['spectra'].sum(axis=0)
        luminosity = integ.simps(summed_spectrum, stars['lambda'])
        luminosities.append(luminosity)
        ion_lums.append(ion_lum)
        del stars
    df[('luminosity',0)] = np.nan
    df[('ion_lum',0)] = np.nan
    return

Units: <br>
all masses: 10^10M_sun/h <br>
SFR: M_sun/h/s <br>
Radius: ckpc/h <br>
Luminosity: L_sun <br>

In [13]:
def build_new_df(df, path, z, h):
    new_df = pd.DataFrame()
    new_df['HaloMass'] = df[('SubhaloMassInRad',0)]
    new_df['GasMass'] = df[('SubhaloMassInRadType',0)]
    new_df['StarMass'] = df[('SubhaloMassInRadType',4)]
    new_df['SFR'] = df[('SubhaloSFRinRad',0)]
    new_df['com_radius'] = 2*df[('SubhaloHalfmassRadType',4)]
    new_df['Z'] = df[('SubhaloGasMetallicity',0)]
    new_df['ion_lum'] = df[('ion_lum',0)]
    new_df['luminosity'] = df[('luminosity',0)]
    
    new_df['radius'] = (z+1)*new_df[com_radius]/h
    new_df['gas_surface_dens'] = 1e10*new_df['GasMass']/(h*np.pi*new_df['radius']**2)
    new_df['star_surface_dens'] = 1e10*new_df['StarMass']/(h*np.pi*new_df['radius']**2)
    new_df['sfr_surface_dens'] = new_df['SFR']/(np.pi*new_df['radius']**2)
    new_df['ionizing_flux'] = new_df['ion_lum']/(np.pi*new_df['radius']**2)
    new_df.to_pickle(path)
    return

In [48]:
df.drop(index=874235, inplace=True)

In [52]:
z_start

6.0107573988449

In [46]:
start = time.time()
idx = 874235
stars = il.snapshot.loadSubhalo(sim_path, snap_num, idx,'stars')
print(stars['count'])
utils.dropWindParticles(stars)
stars['Redshift'] = z_start
stars['ages'] = spectra._computeAgeFromFormationTime(stars['Redshift'],stars['GFM_StellarFormationTime'])
print(stars['count'])
get_stellar_dist(stars, df, idx)
idces = stars['rel_dist']<1
utils._keepPartsByIdx(stars, idces)
print(stars['count'])
print(stars)
#specFac.computeStellarSpectra(stars, Q_0=True)

3
3
0
{'count': 0, 'BirthPos': array([], shape=(0, 3), dtype=float32), 'BirthVel': array([], shape=(0, 3), dtype=float32), 'Coordinates': array([], shape=(0, 3), dtype=float64), 'GFM_InitialMass': array([], dtype=float32), 'GFM_Metallicity': array([], dtype=float32), 'GFM_Metals': array([], shape=(0, 10), dtype=float32), 'GFM_MetalsTagged': array([], shape=(0, 6), dtype=float32), 'GFM_StellarFormationTime': array([], dtype=float32), 'GFM_StellarPhotometrics': array([], shape=(0, 8), dtype=float32), 'Masses': array([], dtype=float32), 'ParticleIDs': array([], dtype=uint64), 'Potential': array([], dtype=float32), 'StellarHsml': array([], dtype=float32), 'SubfindDMDensity': array([], dtype=float32), 'SubfindDensity': array([], dtype=float32), 'SubfindHsml': array([], dtype=float32), 'SubfindVelDisp': array([], dtype=float32), 'TimeStep': array([], dtype=float32), 'Velocities': array([], shape=(0, 3), dtype=float32), 'Redshift': 6.0107573988449, 'ages': array([], dtype=float64), 'rel_dist'

In [15]:
def star_batches(stars, batchsize):
    stars_arr = []
    batches = stars['count']//batchsize+1
    start = 0
    end = batchsize
    for i in range(batches):
        star_batch = {}
        star_batch['Redshift'] = stars['Redshift']
        if i+1 < batches:
            star_batch['count'] = batchsize
            for key in stars.keys():
                if key not in ['count', 'Redshift']:
                    star_batch[key] = stars[key][start:end]
            stars_arr.append(star_batch)
            start = end
            end += batchsize
        else:
            for key in stars.keys():
                if key not in ['count', 'Redshift']:
                    star_batch[key] = stars[key][start:]
            star_batch['count'] = len(stars['ages'][start:])
            stars_arr.append(star_batch)
    return stars_arr

In [16]:
def calculate_batched_quants(stars_arr):
    ion_lum = 0
    luminosity = 0
    for batch in stars_arr:
        specFac.computeStellarSpectra(batch, Q_0=True)
        ion_lum += batch['Q_0'].sum()
        summed_spectrum = batch['spectra'].sum(axis=0)
        luminosity += integ.simps(summed_spectrum, batch['lambda'])
        del batch['spectra']
    return ion_lum, luminosity

In [17]:
start = time.time()
idx = 0
stars = il.snapshot.loadSubhalo(sim_path, snap_num, idx,'stars')
print(stars['count'])
utils.dropWindParticles(stars)
stars['Redshift'] = z_start
stars['ages'] = spectra._computeAgeFromFormationTime(stars['Redshift'],stars['GFM_StellarFormationTime'])
get_stellar_dist(stars, df, idx)
idces = stars['rel_dist']<1
utils._keepPartsByIdx(stars, idces)
if stars['count']<10000:
    specFac.computeStellarSpectra(stars, Q_0=True)
    ion_lum = stars['Q_0'].sum()
    summed_spectrum = stars['spectra'].sum(axis=0)
    luminosity = integ.simps(summed_spectrum, stars['lambda'])
else:
    print(stars['count'])
    star_arr = star_batches(stars, batchsize=10000)
    ion_lum, luminosity = calculate_batched_quants(star_arr)
df.loc[idx]['luminosity'] = luminosity
df.loc[idx]['ion_lum'] = ion_lum
del stars
del star_arr
print(time.time()-start)

646048
412295


/freya/u/ivkos/pybpass/pyBPASS/database.py:137: UserWarning: Input metallicities for interpolation outside of available range 1e-05, 0.04 provided. They will be clipped.
  _warnings.warn(
/freya/u/ivkos/pybpass/pyBPASS/database.py:149: UserWarning: Input ages for interpolation outside of available range 1000000.0, 100000000000.0 [yr] provided. They will be clipped.
  _warnings.warn(


467.69380164146423


<ipython-input-17-fd298f676400>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[idx]['luminosity'] = luminosity
/u/ivkos/conda-envs/crash_analysis/lib/python3.8/site-packages/pandas/core/indexing.py:671: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)
<ipython-input-17-fd298f676400>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[idx]['ion_lum'] = ion_lum
